# Notebook 02 - Feature Engineering

Agora vamos transformar dados brutos em inteligência para o modelo.
Aqui é onde o modelo deixa de ser "ok" e começa a ficar "absurdamente bom".

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

## Carregando dataset processado da EDA

In [2]:
df = pd.read_csv("../data/processed/ev_adoption_processed.csv")
df.head()

,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index,Date,High_EV_Adoption
0,2018,1,Europe,3 Series,7822,47482,371404204,0.011,19.12,3.5,1.0,2018-01-01,0
1,2018,1,Europe,5 Series,10280,61685,634121800,0.019,19.12,3.5,1.0,2018-01-01,0
2,2018,1,Europe,X3,3105,58433,181434465,0.022,19.12,3.5,1.0,2018-01-01,0
3,2018,1,Europe,X5,7420,67955,504226100,0.021,19.12,3.5,1.0,2018-01-01,0
4,2018,1,Europe,X7,8474,92300,782150200,0.035,19.12,3.5,1.0,2018-01-01,0


## Preparando variável temporal
Ordenar por data é CRÍTICO para séries temporais

In [3]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

## Transformação cíclica do mês

Janeiro não está "longe" de Dezembro — modelo precisa entender isso.

In [4]:
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)

## Lag Features

Estamos dizendo:
"o passado influencia o futuro"

In [5]:
df["BEV_Share_lag1"] = df.groupby("Region")["BEV_Share"].shift(1)
df["BEV_Share_lag2"] = df.groupby("Region")["BEV_Share"].shift(2)

## Rolling Mean

Captura tendência recente de adoção

In [6]:
df["BEV_Share_rolling3"] = (
    df.groupby("Region")["BEV_Share"]
    .transform(lambda x: x.rolling(3).mean())
)

## Transformando variáveis categóricas
Modelos não entendem texto

In [7]:
df = pd.get_dummies(df, columns=["Region"], drop_first=True)

## Removendo valores nulos
Gerados pelas primeiras linhas das lags

In [8]:
df = df.dropna()

## Separando features e target

In [9]:
X = df.drop(columns=["High_EV_Adoption", "Date"])
y = df["High_EV_Adoption"]

print(X.shape, y.shape)

(3064, 18) (3064,)


## Salvando dataset final para modelagem

In [10]:
X.to_csv("../data/processed/X.csv", index=False)
y.to_csv("../data/processed/y.csv", index=False)

## Conclusões

- Criamos variáveis temporais inteligentes
- Introduzimos memória no modelo (lags)
- Capturamos tendência (rolling)
- Transformamos dados categóricos
- Dataset pronto para Machine Learning

Próximo passo: Treinamento e avaliação de modelos